Importation des bibliothèques

In [30]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torch.optim as optim
from torch.utils.data import random_split
from tqdm import tqdm # pour la progression 
import time #pour le temps de calcul
import copy #pour copier base_model

In [31]:
print(os.getcwd())  # donne le répertoire courant

d:\INRIA\MCDropout


Configuration de base

Importation du modèle déjà entraîné par l'utilisateur

In [32]:
batch_size = 128 #on peut changer la taille mais 128 est bien pour éviter underfitting ou overfitting

transform = transforms.Compose([
    transforms.ToTensor(),                   #Convertir une image en tenseur, met les valeurs des pixels entre 0 et 1
    transforms.Normalize((0.5, 0.5, 0.5),    # Moyenne RGB
                         (0.5, 0.5, 0.5))])  # Écart-type RGB; pixels deviennent centrés autour de 0

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                        download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')


Définition du CNN de base (3 couches)

In [33]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)  # entrée 3 channels (RGB)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)  # réduit dimension par 2 à chaque fois
        
        # Calculer la taille en sortie avant les couches fully connected
        # CIFAR10 32x32 → après 3 poolings (x2) → 4x4
        self.fc1 = nn.Linear(64 * 4 * 4, 128) # après les convolutions on a un tenseur de taille 64*4*4, fc1 les réduit en 128 neurones
        self.fc2 = nn.Linear(128, 10)  # 10 classes CIFAR10

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        
        x = F.relu(self.conv3(x))
        x = self.pool(x)
        
        x = x.view(x.size(0), -1)  # aplatit en [batch_size, 64*4*4 = 1024]
        x = F.relu(self.fc1(x)) #ou fonction sigmoïde à la fin?
        x = self.fc2(x)
        return x


à entraîner (boucle d'entraînement) training loop + sauvegarde des poids à chaque epoch ; tester pour chaque epoch et voir où l'accuracy est la mieux

Masque pour le dropout

In [34]:
class CNN_MCdropout(nn.Module):
    def __init__(self, model, mc_layers=None, p1=0.2, p2=0.2, p3=0.2, p4=0.2):
        super().__init__()
        self.model = model
        self.mc_layers = mc_layers or []

        # Ajout dropout après conv1
        if 'conv1' in self.mc_layers and isinstance(self.model.conv1, nn.Conv2d):
            self.model.conv1 = nn.Sequential(self.model.conv1, nn.ReLU(), nn.Dropout2d(p1))
        else:
            self.model.conv1 = nn.Sequential(self.model.conv1, nn.ReLU())

        # Ajout dropout après conv2
        if 'conv2' in self.mc_layers and isinstance(self.model.conv2, nn.Conv2d):
            self.model.conv2 = nn.Sequential(self.model.conv2, nn.ReLU(), nn.Dropout2d(p2))
        else:
            self.model.conv2 = nn.Sequential(self.model.conv2, nn.ReLU())

        # Ajout dropout après conv3
        if 'conv3' in self.mc_layers and isinstance(self.model.conv3, nn.Conv2d):
            self.model.conv3 = nn.Sequential(self.model.conv3, nn.ReLU(), nn.Dropout2d(p3))
        else:
            self.model.conv3 = nn.Sequential(self.model.conv3, nn.ReLU())

        # Ajout dropout après fc1
        if 'fc1' in self.mc_layers and isinstance(self.model.fc1, nn.Linear):
            self.model.fc1 = nn.Sequential(self.model.fc1, nn.ReLU(), nn.Dropout(p4))
        else:
            self.model.fc1 = nn.Sequential(self.model.fc1, nn.ReLU())

    def forward(self, x):
        x = self.model.conv1(x)
        x = self.model.pool(x)

        x = self.model.conv2(x)
        x = self.model.pool(x)

        x = self.model.conv3(x)
        x = self.model.pool(x)

        x = x.view(x.size(0), -1)
        x = self.model.fc1(x)
        x = self.model.fc2(x)
        return x

Fonction d'entraînement et test

In [35]:
val_ratio = 0.1  # 10% pour validation
train_size = int((1 - val_ratio) * len(trainset))
val_size   = len(trainset) - train_size

train_subset, val_subset = random_split(trainset, [train_size, val_size]) # divise le dataset de manière aléatoire en 90/10

trainloader = torch.utils.data.DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
valloader   = torch.utils.data.DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)

Fonction d'évaluation (avant entraînement)

In [36]:
def evaluate(model, dataloader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            total_loss += criterion(outputs, targets).item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    return total_loss/len(dataloader), correct/total

Fonction d'entraînement

In [37]:
def train_model(model, trainloader, valloader, device, epochs=20, save_path="best.pt"):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_val_acc = 0
    
    for epoch in range(epochs):#petit nombre d'epochs pour tester (environ 1 minutes pas epoch)
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        # Évaluer sur validation
        val_loss, val_acc = evaluate(model, valloader, device)
        print(f"Epoch {epoch+1}/{epochs} - Train Loss: {running_loss/len(trainloader):.4f} - Val Loss: {val_loss:.4f} - Val Acc: {val_acc:.4f}")
        
        # Sauvegarde si amélioration
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), save_path)
    
    print("Finished Training")
    return model

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
save_path = "best.pt"

# Demande à l'utilisateur quelles couches doivent subir le dropout : ou alors il amène sa liste
user_layers = input(
    "Sur quelles couches voulez-vous appliquer le MC Dropout ? "
    "(choisissez parmi conv1, conv2, conv3, fc1, séparées par des virgules) : ")
mc_layers = [layer.strip() for layer in user_layers.split(',') if layer.strip() in ['conv1','conv2','conv3','fc1']]

# Vérifie si les poids existent déjà
base_model = SimpleCNN()
if os.path.exists(save_path):
    print("Chargement du modèle sauvegardé")
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # même architecture que celle qui a sauvegardé
else:
    print("Pas de modèle sauvegardé, on entraîne le modèle")
    base_model = train_model(base_model, trainloader, valloader, device, epochs=20, save_path=save_path)
    base_model.load_state_dict(torch.load(save_path, map_location=device))  # recharge les meilleurs poids

# On fait une copie pour MC Dropout
base_model_mc = copy.deepcopy(base_model)

model = CNN_MCdropout(base_model_mc, mc_layers=mc_layers, p1=0.2, p2=0.2, p3=0.2, p4=0.2).to(device)

# Évaluation finale
test_loss, test_acc = evaluate(model, testloader, device)
print(f"Final Test Loss: {test_loss:.4f} - Test Acc: {test_acc:.4f}")

Chargement du modèle sauvegardé
Final Test Loss: 0.8186 - Test Acc: 0.7267


MC Dropout prédiction

In [39]:
def mc_predict_mean_probs(model, X, T=1000, verbose=True):
    model.train()  # activer dropout pendant l'inférence MC
    probs_list = []
    start_time = time.time() 
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            logits_t = model(X)
            probs_t = F.softmax(logits_t, dim=1)
            probs_list.append(probs_t.unsqueeze(0))
    elapsed = time.time() - start_time
    probs_mc = torch.cat(probs_list, dim=0)       
    model.eval()  # remettre le modèle en mode eval à la fin
    if verbose:
        print(f"Temps total: {elapsed:.2f} s  |  Temps moyen par passe: {elapsed/T:.4f} s")
    return probs_mc.mean(0), elapsed                        

je dois garder le même batch ; mettre des seeds pour que ce soit reproductible (fonction de dropout)

In [40]:
# Test MC Dropout sur un batch
X, Y = next(iter(testloader))
X, Y = X.to(device), Y.to(device)
probs, t1 = mc_predict_mean_probs(model, X, T=1000, verbose=True)
Y_hat = probs.argmax(1)

print("Classes vraies       :", Y.tolist())
print(f"Classes prédites   (t={t1:.2f}s): {Y_hat.tolist()}")


100%|██████████| 1000/1000 [00:12<00:00, 81.77it/s]


Temps total: 12.24 s  |  Temps moyen par passe: 0.0122 s
Classes vraies       : [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 0, 4, 9, 5, 2, 4, 0, 9, 6, 6, 5, 4, 5, 9, 2, 4, 1, 9, 5, 4, 6, 5, 6, 0, 9, 3, 9, 7, 6, 9, 8, 0, 3, 8, 8, 7, 7, 4, 6, 7, 3, 6, 3, 6, 2, 1, 2, 3, 7, 2, 6, 8, 8, 0, 2, 9, 3, 3, 8, 8, 1, 1, 7, 2, 5, 2, 7, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 7, 4, 5, 6, 3, 1, 1, 3, 6, 8, 7, 4, 0, 6, 2, 1, 3, 0, 4, 2, 7, 8, 3, 1, 2, 8, 0, 8, 3]
Classes prédites   (t=12.24s): [3, 8, 8, 0, 6, 6, 1, 6, 3, 1, 0, 9, 5, 7, 9, 8, 5, 7, 8, 6, 7, 2, 4, 1, 4, 2, 4, 0, 9, 6, 3, 5, 2, 3, 9, 3, 4, 1, 9, 5, 4, 6, 3, 6, 0, 9, 3, 3, 7, 2, 9, 8, 2, 3, 8, 8, 5, 5, 5, 3, 7, 5, 6, 3, 6, 2, 1, 0, 3, 7, 0, 6, 8, 8, 8, 2, 2, 3, 3, 8, 8, 1, 1, 7, 2, 2, 2, 8, 8, 9, 0, 3, 8, 6, 4, 6, 6, 0, 0, 3, 4, 5, 6, 3, 1, 1, 2, 6, 8, 5, 6, 0, 2, 2, 9, 3, 0, 4, 3, 5, 8, 2, 1, 2, 8, 0, 8, 3]


rajouter en argument de generate_mc_outputs measure = variance (en gros on laisse à l'utilisateur le choix de la mesure d'incertitude)

In [41]:
def generate_mc_outputs(model, X, T=1000, metrics=["mc_estimate"], labels=None, verbose=True):
    model.train()
    outputs = []

    start_forward = time.time()
    with torch.no_grad():
        for _ in tqdm(range(T), disable=not verbose):
            out = model(X)
            outputs.append(out.unsqueeze(0))
    elapsed_forward = time.time() - start_forward

    outputs = torch.cat(outputs, dim=0)
    results = {}
    elapsed_metrics = {}

    for metric in metrics:
        start_metric = time.time()
        if metric == "mc_estimate":
            all_probs = torch.softmax(outputs, dim=2)
            results[metric] = all_probs.mean(dim=0)
        elif metric == "variance":
            results[metric] = outputs.var(dim=0)
        elif metric == "predictive_entropy":
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            results[metric] = -(mean_probs * (mean_probs + 1e-12).log()).sum(dim=1)
        elif metric == "relative_norm":
            if labels is None:
                raise ValueError("labels doivent être fournis pour relative_norm")
            all_probs = torch.softmax(outputs, dim=2)
            mean_probs = all_probs.mean(dim=0)
            labels_onehot = F.one_hot(labels, num_classes=mean_probs.size(1)).float()
            diff_norm = torch.norm(mean_probs - labels_onehot, dim=1)
            denom = torch.max(torch.norm(mean_probs, dim=1), torch.norm(labels_onehot, dim=1))
            results[metric] = diff_norm / (denom + 1e-12)
        else:
            raise ValueError(f"Métrique {metric} non reconnue")
        elapsed_metrics[metric] = time.time() - start_metric

    model.eval()

    if verbose:
        print(f"Temps forward pass: {elapsed_forward:.2f}s  |  Temps moyen par passe: {elapsed_forward/T:.4f}s")
        for m, t in elapsed_metrics.items():
            print(f"Temps calcul métrique '{m}': {t:.6f}s")

    return outputs, mean_probs, results, elapsed_forward, elapsed_metrics


le mc estimate n'est pas une mesure d'incertitude, sert pour construire une autre variable

Métriques

In [42]:
user_metrics = input(
    "Quelles métriques voulez-vous calculer ? (mc_estimate, variance, predictive_entropy, relative_norm)\n"
    "Vous pouvez en choisir plusieurs, séparées par des virgules : ")
user_metrics = [m.strip() for m in user_metrics.split(",")]
outputs, mean_probs, metric_values, elapsed_forward, elapsed_metrics = generate_mc_outputs(model, X, T=1000, metrics=user_metrics, labels=Y, verbose=False)

print(f"Liste des métriques choisies par l\'utilisateur : {user_metrics}")
print(f"Temps total des forward passes : {elapsed_forward:.2f} s")
for metric in user_metrics:
    print(f"Métrique choisie : {metric}")
    print(f"Temps de calcul de la métrique : {elapsed_metrics[metric]:.6f} s")
    print(f"Résultat : {metric_values[metric]}\n")

Liste des métriques choisies par l'utilisateur : ['mc_estimate', 'variance', 'predictive_entropy', 'relative_norm']
Temps total des forward passes : 14.89 s
Métrique choisie : mc_estimate
Temps de calcul de la métrique : 0.003642 s
Résultat : tensor([[0.0604, 0.0411, 0.0470,  ..., 0.0091, 0.1686, 0.0351],
        [0.0566, 0.3304, 0.0125,  ..., 0.0037, 0.5124, 0.0163],
        [0.1747, 0.0995, 0.0578,  ..., 0.0355, 0.3550, 0.0545],
        ...,
        [0.3897, 0.0905, 0.2169,  ..., 0.0052, 0.1048, 0.0696],
        [0.2446, 0.0180, 0.1275,  ..., 0.0088, 0.3001, 0.0174],
        [0.0300, 0.0144, 0.0665,  ..., 0.0400, 0.0211, 0.0402]])

Métrique choisie : variance
Temps de calcul de la métrique : 0.007044 s
Résultat : tensor([[11.9160, 17.5779,  8.0808,  ..., 13.0830, 19.7641, 23.2949],
        [13.0963, 26.2571, 14.6597,  ..., 30.5814, 23.7661, 16.5589],
        [ 6.3217, 15.1695,  7.1012,  ..., 10.2959, 10.7780, 11.5868],
        ...,
        [13.2521, 19.2627, 14.0505,  ..., 24.1839, 1

pour la fonction de confiance, dois-je faire 1/métrique choisie? ou ça marche qu'avec la variance?

oui si c'est dans R+ : 1/ ? exp(-...) si c'est pour avoir entre 0 et 1 ; quitte à normaliser après